# 03 — Quantitative analysis

Uses the restricted cleaned dataset produced by Notebook 01. Preserves the original tests, bootstrap procedures (10,000 percentile resamples, seed 42), centered OLS moderation model, HC3 sensitivity check, diagnostics and simple slopes. Checks sample sizes, variable ranges and reported estimates. No raw participant data should be committed to GitHub. Run from a fresh kernel and inspect all outputs before publication.

# Setup and input validation

In [ ]:
# Import libraries
# This notebook contains the formal inferential statistical
# analyses used to address the quantitative research questions.
#
# Unlike 02_exploration.ipynb, which examined distributions and
# preliminary patterns, this notebook performs the statistical
# tests used for thesis reporting.
#
# The analysis focuses on:
#
#   RQ1:
#   How is anxiety related to students' use of LLMs in
#   coding-related tasks?
#
#   RQ2:
#   Does AI self-efficacy moderate the relationship between
#   anxiety and LLM use?
#
# The processed dataset generated in 01_cleaning.ipynb is used
# as the sole data source.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [ ]:
# Load processed dataset
# The cleaned and scored survey dataset is loaded from the
# processed-data directory.
#
# The raw survey export is not used directly in this notebook.
# This ensures that all formal analyses are based on the same
# verified dataset produced by 01_cleaning.ipynb.

from pathlib import Path
import pandas as pd

# Locate the project root
# Works from notebooks, notebooks_clean, notebooks_review,
# or the main project directory.

current_directory = Path.cwd().resolve()

project_root = next(
    (
        folder
        for folder in (current_directory, *current_directory.parents)
        if (
            folder / "data" / "processed" / "survey_processed.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Cannot locate data/processed/survey_processed.csv "
        f"from {current_directory}.\n"
        "Check that Notebook 01 completed successfully."
    )

# Define the processed-data file

processed_file = (
    project_root
    / "data"
    / "processed"
    / "survey_processed.csv"
)

# Confirm that the file exists

if not processed_file.is_file():
    raise FileNotFoundError(
        f"Processed dataset not found:\n{processed_file}\n\n"
        "Run 01_cleaning.ipynb before continuing."
    )

# Load the dataset

df = pd.read_csv(processed_file)

print("Project root:", project_root)
print("Processed dataset loaded successfully.")
print(f"\nDataset dimensions: {df.shape}")

In [ ]:
# Define and verify analysis variables
# The main variables are defined explicitly before formal
# statistical testing.
#
# Primary variables:
#
#   gad7_total
#       Continuous GAD-7 anxiety symptom score (0-21).
#
#   aise_mean
#       Mean AI self-efficacy score across six items (1-7).
#
#   llm_overall
#       Single ordinal item measuring overall LLM-use frequency
#       from 1 (Never) to 5 (Always).
#
#   llm_task_mean
#       Mean frequency of LLM use across five coding-related
#       tasks (1-5).
#
# Individual coding-task variables are retained for secondary
# task-specific analyses.


core_variables = [
    "gad7_total",
    "aise_mean",
    "llm_overall",
    "llm_task_mean"
]


llm_task_variables = [
    "llm_debug",
    "llm_errors",
    "llm_generate",
    "llm_refactor",
    "llm_validate"
]


required_variables = (
    core_variables
    + llm_task_variables
)


# Check that all required variables exist

missing_variables = [
    variable
    for variable in required_variables
    if variable not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Required variables are missing: "
        f"{missing_variables}"
    )


# Check missing data

missing_values = (
    df[required_variables]
    .isna()
    .sum()
)

print("Number of participants:")
print(len(df))

print("\nMissing values in analysis variables:")
print(missing_values)

print("\nAnalysis variables verified successfully.")

# Fail early if the analysis sample or measurement scales differ from the study.
if len(df) != 124:
    raise ValueError(f"Expected 124 observations, found {len(df)}")
if missing_values.any():
    raise ValueError(f"Missing analysis values: {missing_values[missing_values > 0].to_dict()}")
for variable, lower, upper in [
    ("gad7_total", 0, 21), ("aise_mean", 1, 7),
    ("llm_overall", 1, 5), ("llm_task_mean", 1, 5),
    *[(task, 1, 5) for task in llm_task_variables],
]:
    values = pd.to_numeric(df[variable], errors="coerce")
    if values.isna().any() or not values.between(lower, upper).all():
        raise ValueError(f"Invalid values in {variable}")
if not np.allclose(df[llm_task_variables].mean(axis=1), df["llm_task_mean"], atol=1e-8):
    raise ValueError("llm_task_mean does not equal the five-task average")
print("PASS: complete N=124 analysis sample, scales, and task mean")

# RQ1: primary correlations and bootstrap intervals

In [ ]:
# Rq1 — anxiety and llm use
# RQ1 asks whether students' anxiety symptoms are associated
# with their use of LLMs in coding-related activities.
#
# Two representations of LLM use are examined:
#
#   1. llm_overall
#      A single five-category ordinal frequency item.
#
#   2. llm_task_mean
#      A mean score across five coding-specific LLM-use items.
#
# Spearman's rank correlation is used as the primary method for
# llm_overall because this variable is ordinal.
#
# For llm_task_mean, both Pearson and Spearman correlations are
# calculated. Comparing the two helps determine whether the
# substantive conclusion depends strongly on the correlation
# method.
#
# Statistical significance is evaluated using p-values, but
# interpretation will also consider the direction, magnitude,
# and later confidence intervals of the associations.


# RQ1A: GAD-7 and overall LLM-use frequency

rho_overall, p_overall = stats.spearmanr(
    df["gad7_total"],
    df["llm_overall"]
)


# RQ1B: GAD-7 and mean coding-task LLM use — Pearson

r_task_pearson, p_task_pearson = stats.pearsonr(
    df["gad7_total"],
    df["llm_task_mean"]
)


# RQ1B: GAD-7 and mean coding-task LLM use — Spearman

rho_task, p_task_spearman = stats.spearmanr(
    df["gad7_total"],
    df["llm_task_mean"]
)


# Organise results into a readable table

rq1_results = pd.DataFrame({

    "Relationship": [
        "GAD-7 ↔ Overall LLM use",
        "GAD-7 ↔ Mean coding-task LLM use",
        "GAD-7 ↔ Mean coding-task LLM use"
    ],

    "Method": [
        "Spearman",
        "Pearson",
        "Spearman"
    ],

    "Coefficient": [
        rho_overall,
        r_task_pearson,
        rho_task
    ],

    "p-value": [
        p_overall,
        p_task_pearson,
        p_task_spearman
    ]
})


# Add a simple significance indicator for readability
# This indicator is descriptive only. Statistical interpretation
# should not be reduced to a binary significant/non-significant
# decision.

rq1_results["p < .05"] = (
    rq1_results["p-value"] < 0.05
)


print("============================================================")
print("              RQ1 PRIMARY CORRELATION RESULTS               ")
print("============================================================")

display(
    rq1_results.round(4)
)

In [ ]:
# Calculate 95% bootstrap confidence intervals for rq1
# A correlation coefficient is an estimate based on the current
# sample. A confidence interval provides information about the
# uncertainty surrounding that estimate.
#
# Bootstrap confidence intervals are calculated by repeatedly
# resampling participants WITH replacement and recalculating the
# correlation in each resampled dataset.
#
# Each bootstrap sample contains the same number of participants
# as the original dataset, but some participants may appear more
# than once while others may not appear in a particular sample.
#
# Repeating this process many times approximates the sampling
# distribution of the correlation coefficient.
#
# The 2.5th and 97.5th percentiles of the resulting bootstrap
# distribution are used as the bounds of the 95% confidence
# interval.
#
# A fixed random seed is used so that the results are
# reproducible when the notebook is rerun.


# Function for bootstrap correlation confidence intervals

def bootstrap_correlation_ci(
    data,
    x_variable,
    y_variable,
    method="spearman",
    n_bootstrap=10000,
    confidence_level=0.95,
    random_seed=42
):
    """
    Calculate a percentile bootstrap confidence interval for a
    Pearson or Spearman correlation.

    Parameters
    ----------
    data : pandas DataFrame
        Dataset containing the variables.

    x_variable : str
        Name of the first variable.

    y_variable : str
        Name of the second variable.

    method : str
        Either "pearson" or "spearman".

    n_bootstrap : int
        Number of bootstrap resamples.

    confidence_level : float
        Confidence level used for the interval.

    random_seed : int
        Fixed seed for reproducibility.

    Returns
    -------
    tuple
        Lower and upper confidence-interval bounds.
    """

    # Keep only the two variables required for this analysis
    # and remove incomplete pairs if any exist.
    paired_data = (
        data[[x_variable, y_variable]]
        .dropna()
        .reset_index(drop=True)
    )

    n = len(paired_data)

    rng = np.random.default_rng(
        random_seed
    )

    bootstrap_correlations = []

    for _ in range(n_bootstrap):

        # Sample participant row indices with replacement.
        bootstrap_indices = rng.integers(
            0,
            n,
            size=n
        )

        bootstrap_sample = paired_data.iloc[
            bootstrap_indices
        ]

        x = bootstrap_sample[x_variable]
        y = bootstrap_sample[y_variable]

        if method == "pearson":

            coefficient, _ = stats.pearsonr(
                x,
                y
            )

        elif method == "spearman":

            coefficient, _ = stats.spearmanr(
                x,
                y
            )

        else:

            raise ValueError(
                "Method must be either "
                "'pearson' or 'spearman'."
            )

        bootstrap_correlations.append(
            coefficient
        )


    # --------------------------------------------------------
    # Determine percentile boundaries
    # --------------------------------------------------------

    alpha = 1 - confidence_level

    lower_percentile = (
        100 * alpha / 2
    )

    upper_percentile = (
        100 * (1 - alpha / 2)
    )


    lower_ci = np.percentile(
        bootstrap_correlations,
        lower_percentile
    )

    upper_ci = np.percentile(
        bootstrap_correlations,
        upper_percentile
    )

    return lower_ci, upper_ci

In [ ]:
# Calculate rq1 bootstrap confidence intervals
# The bootstrap function defined above is now applied to the
# three primary RQ1 correlation estimates.
#
# The resulting 95% confidence intervals are added to the
# existing rq1_results table created in Step 4.
#
# This allows the correlation coefficient, uncertainty, and
# p-value to be interpreted together rather than relying only
# on statistical significance.


# Calculate RQ1 confidence intervals

ci_overall = bootstrap_correlation_ci(
    df,
    "gad7_total",
    "llm_overall",
    method="spearman"
)

ci_task_pearson = bootstrap_correlation_ci(
    df,
    "gad7_total",
    "llm_task_mean",
    method="pearson"
)

ci_task_spearman = bootstrap_correlation_ci(
    df,
    "gad7_total",
    "llm_task_mean",
    method="spearman"
)


# Add confidence intervals to the RQ1 results table

rq1_results["95% CI Lower"] = [
    ci_overall[0],
    ci_task_pearson[0],
    ci_task_spearman[0]
]

rq1_results["95% CI Upper"] = [
    ci_overall[1],
    ci_task_pearson[1],
    ci_task_spearman[1]
]


# Reorder columns for easier interpretation

rq1_results = rq1_results[
    [
        "Relationship",
        "Method",
        "Coefficient",
        "95% CI Lower",
        "95% CI Upper",
        "p-value",
        "p < .05"
    ]
]


# Display the completed RQ1 results table

print("============================================================")
print("        RQ1 RESULTS WITH 95% CONFIDENCE INTERVALS            ")
print("============================================================")

display(
    rq1_results.round(4)
)

# RQ1: task-specific tests and Holm correction

In [ ]:
# Rq1 — task-specific anxiety and llm-use relationships
# The mean LLM task-frequency score provides an overall summary,
# but it may conceal differences between individual coding tasks.
#
# RQ1 is therefore examined at the task level by testing the
# association between GAD-7 scores and each of the five
# coding-specific LLM-use variables.
#
# Each task-frequency item is measured using a five-category
# ordinal scale from Never (1) to Always (5).
#
# Spearman's rank correlation is therefore used for these
# task-specific analyses.
#
# Because five related statistical tests are performed, the
# issue of multiple comparisons will be addressed separately in
# the following step.


task_results = []


for variable in llm_task_variables:

    rho, p_value = stats.spearmanr(
        df["gad7_total"],
        df[variable]
    )

    task_results.append({
        "Task variable": variable,
        "Spearman rho": rho,
        "p-value": p_value
    })


rq1_task_results = pd.DataFrame(
    task_results
)


# Add human-readable task names

task_labels = {
    "llm_debug":
        "Debugging code",

    "llm_errors":
        "Understanding error messages",

    "llm_generate":
        "Generating code snippets",

    "llm_refactor":
        "Refactoring/improving code",

    "llm_validate":
        "Editing/validating AI-generated code"
}


rq1_task_results.insert(
    0,
    "Coding task",
    rq1_task_results[
        "Task variable"
    ].map(task_labels)
)


print("============================================================")
print("           RQ1 TASK-SPECIFIC CORRELATIONS                   ")
print("============================================================")

display(
    rq1_task_results.round(4)
)

In [ ]:
# Correct task-specific p-values for multiple comparisons
# Five separate task-specific correlations were calculated in
# Step 6. Performing several statistical tests increases the
# probability of obtaining at least one small p-value by chance.
#
# The Holm procedure is therefore applied to the five p-values.
#
# Holm correction controls the family-wise error rate while
# generally being less conservative than the traditional
# Bonferroni correction.
#
# Both raw and Holm-adjusted p-values are retained:
#
#   Raw p-value
#       Result of the individual Spearman correlation.
#
#   Holm-adjusted p-value
#       P-value after accounting for the family of five
#       task-specific tests.
#
# Formal conclusions about the task-specific analyses should be
# based primarily on the adjusted p-values.


reject_holm, p_holm, _, _ = multipletests(
    rq1_task_results["p-value"],
    alpha=0.05,
    method="holm"
)


rq1_task_results["Holm-adjusted p"] = p_holm

rq1_task_results["Significant after Holm"] = (
    reject_holm
)


# Reorder columns for easier interpretation.
rq1_task_results = rq1_task_results[
    [
        "Coding task",
        "Task variable",
        "Spearman rho",
        "p-value",
        "Holm-adjusted p",
        "Significant after Holm"
    ]
]


print("============================================================")
print("       RQ1 TASK-SPECIFIC RESULTS — HOLM CORRECTION           ")
print("============================================================")

display(
    rq1_task_results.round(4)
)

In [ ]:
# Add 95% bootstrap confidence intervals to task results
# Confidence intervals are calculated for each task-specific
# Spearman correlation using the bootstrap procedure defined
# earlier.
#
# This allows the magnitude and uncertainty of each association
# to be evaluated alongside both the raw and Holm-adjusted
# p-values.


task_ci_lower = []
task_ci_upper = []


for variable in rq1_task_results["Task variable"]:

    lower_ci, upper_ci = bootstrap_correlation_ci(
        df,
        "gad7_total",
        variable,
        method="spearman"
    )

    task_ci_lower.append(lower_ci)
    task_ci_upper.append(upper_ci)


rq1_task_results["95% CI Lower"] = (
    task_ci_lower
)

rq1_task_results["95% CI Upper"] = (
    task_ci_upper
)


# Reorder the table.
rq1_task_results = rq1_task_results[
    [
        "Coding task",
        "Spearman rho",
        "95% CI Lower",
        "95% CI Upper",
        "p-value",
        "Holm-adjusted p",
        "Significant after Holm",
        "Task variable"
    ]
]


print("============================================================")
print("       COMPLETE RQ1 TASK-SPECIFIC RESULTS                   ")
print("============================================================")

display(
    rq1_task_results.round(4)
)

# RQ2: centered moderation model

In [ ]:
# Prepare variables for rq2 moderation analysis
# RQ2 examines whether AI self-efficacy moderates the
# relationship between anxiety and coding-task LLM use.
#
# The moderation model uses:
#
#   Predictor (X):
#       gad7_total
#
#   Moderator (M):
#       aise_mean
#
#   Outcome (Y):
#       llm_task_mean
#
# Before constructing the interaction term, GAD-7 and AI
# self-efficacy are mean-centred.
#
# Mean-centering subtracts the sample mean from each score:
#
#       centred score = original score - sample mean
#
# This does NOT change the underlying relationships or the
# interaction effect. It makes the regression coefficients and
# intercept easier to interpret and reduces non-essential
# multicollinearity between the interaction term and its
# component variables.


df["gad7_centered"] = (
    df["gad7_total"]
    - df["gad7_total"].mean()
)

df["aise_centered"] = (
    df["aise_mean"]
    - df["aise_mean"].mean()
)


# Create the interaction term

df["gad7_x_aise"] = (
    df["gad7_centered"]
    * df["aise_centered"]
)


print("Means after centering:")

print(
    df[
        [
            "gad7_centered",
            "aise_centered"
        ]
    ].mean().round(10)
)

print(
    "\nModeration variables prepared successfully."
)

In [ ]:
# Rq2 — moderation regression
# A multiple linear regression model is used to test whether
# AI self-efficacy moderates the relationship between anxiety
# and mean coding-task LLM use.
#
# Model:
#
#   llm_task_mean =
#       b0
#       + b1(gad7_centered)
#       + b2(aise_centered)
#       + b3(gad7_centered × aise_centered)
#       + error
#
# Interpretation:
#
#   b1 = relationship between anxiety and LLM use when AI
#        self-efficacy is at its sample mean.
#
#   b2 = relationship between AI self-efficacy and LLM use
#        when anxiety is at its sample mean.
#
#   b3 = interaction/moderation effect.
#
# The interaction coefficient b3 directly addresses RQ2.
# If b3 differs meaningfully from zero, the anxiety–LLM-use
# relationship changes depending on AI self-efficacy.


moderation_model = smf.ols(
    formula=(
        "llm_task_mean ~ "
        "gad7_centered + "
        "aise_centered + "
        "gad7_x_aise"
    ),
    data=df
).fit()


print(
    moderation_model.summary()
)

In [ ]:
# Create clean rq2 regression results table
# The main regression estimates are extracted into a compact
# table containing:
#
#   - regression coefficient (b)
#   - standard error
#   - t statistic
#   - p-value
#   - 95% confidence interval
#
# Particular attention should be given to the interaction term
# gad7_x_aise, because this coefficient tests moderation.


confidence_intervals = (
    moderation_model.conf_int(
        alpha=0.05
    )
)


rq2_results = pd.DataFrame({

    "Predictor": (
        moderation_model.params.index
    ),

    "b": (
        moderation_model.params.values
    ),

    "SE": (
        moderation_model.bse.values
    ),

    "t": (
        moderation_model.tvalues.values
    ),

    "p-value": (
        moderation_model.pvalues.values
    ),

    "95% CI Lower": (
        confidence_intervals[0].values
    ),

    "95% CI Upper": (
        confidence_intervals[1].values
    )
})


predictor_labels = {
    "Intercept":
        "Intercept",

    "gad7_centered":
        "Anxiety (GAD-7)",

    "aise_centered":
        "AI self-efficacy",

    "gad7_x_aise":
        "Anxiety × AI self-efficacy"
}


rq2_results["Predictor"] = (
    rq2_results["Predictor"]
    .map(predictor_labels)
)


print("============================================================")
print("              RQ2 MODERATION RESULTS                        ")
print("============================================================")

display(
    rq2_results.round(4)
)


print("\nModel R-squared:")
print(
    round(
        moderation_model.rsquared,
        4
    )
)

print("\nAdjusted R-squared:")
print(
    round(
        moderation_model.rsquared_adj,
        4
    )
)

print("\nOverall model p-value:")
print(
    moderation_model.f_pvalue
)

# Regression diagnostics

In [ ]:
# Check moderation regression diagnostics
# Linear regression relies on assumptions concerning the model
# residuals and functional form.
#
# Diagnostics are therefore examined for:
#
#   1. Residual pattern / linearity
#   2. Homoscedasticity
#   3. Residual distribution
#   4. Multicollinearity
#   5. Potentially influential observations
#
# No participant is automatically removed because of a
# diagnostic flag. Unusual observations must first be examined
# and justified substantively.


# Obtain fitted values and residuals

fitted_values = (
    moderation_model.fittedvalues
)

residuals = (
    moderation_model.resid
)


# 12A. Residuals versus fitted values

plt.figure(figsize=(7, 5))

plt.scatter(
    fitted_values,
    residuals,
    alpha=0.6
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title(
    "Residuals vs Fitted Values"
)

plt.tight_layout()
plt.show()


# 12B. Q-Q plot of residuals

plt.figure(figsize=(6, 6))

stats.probplot(
    residuals,
    dist="norm",
    plot=plt
)

plt.title(
    "Q-Q Plot of Regression Residuals"
)

plt.tight_layout()
plt.show()

# 12C. Breusch-Pagan test for heteroscedasticity
# Null hypothesis:
#     residual variance is constant.
#
# A small p-value suggests evidence of heteroscedasticity.

bp_test = het_breuschpagan(
    residuals,
    moderation_model.model.exog
)

bp_labels = [
    "LM statistic",
    "LM p-value",
    "F statistic",
    "F p-value"
]

bp_results = dict(
    zip(
        bp_labels,
        bp_test
    )
)

print("Breusch-Pagan test:")
for key, value in bp_results.items():
    print(
        f"{key}: {value:.4f}"
    )

# 12D. Variance Inflation Factors (VIF)
# VIF indicates how strongly a predictor can be explained by
# the other predictors in the model.
#
# Large values can indicate problematic multicollinearity.
# Mean-centering usually reduces non-essential collinearity
# involving interaction terms.


vif_data = pd.DataFrame()

vif_data["Variable"] = (
    moderation_model.model.exog_names
)

vif_data["VIF"] = [
    variance_inflation_factor(
        moderation_model.model.exog,
        i
    )
    for i in range(
        moderation_model.model.exog.shape[1]
    )
]


print("\nVariance Inflation Factors:")

display(
    vif_data.round(3)
)

# 12E. Cook's distance
# Cook's distance estimates how strongly each observation
# influences the fitted regression model.
#
# The common 4/N guideline is used only as a screening rule,
# not as an automatic deletion criterion.


influence = (
    moderation_model.get_influence()
)

cooks_distance = (
    influence.cooks_distance[0]
)

cooks_threshold = (
    4 / len(df)
)


print(
    f"Cook's distance screening threshold "
    f"(4/N): {cooks_threshold:.4f}"
)


potentially_influential = np.where(
    cooks_distance
    > cooks_threshold
)[0]


print(
    "Number of observations above "
    "the screening threshold:"
)

print(
    len(potentially_influential)
)


print(
    "\nRow indices above threshold:"
)

print(
    potentially_influential
)

# Robustness and conditional effects

In [ ]:
# Robustness check using hc3 robust standard errors
# As a sensitivity analysis, the moderation model is refitted
# using HC3 heteroscedasticity-consistent standard errors.
#
# The regression coefficients themselves do not change.
# HC3 modifies the estimated standard errors, confidence
# intervals, and p-values so that inference is less sensitive
# to heteroscedasticity.
#
# The conventional OLS model remains the primary model, while
# HC3 provides a robustness comparison.


moderation_model_hc3 = (
    moderation_model.get_robustcov_results(
        cov_type="HC3"
    )
)


hc3_confidence_intervals = (
    moderation_model_hc3.conf_int(
        alpha=0.05
    )
)


rq2_hc3_results = pd.DataFrame({

    "Predictor": (
        moderation_model.model.exog_names
    ),

    "b": (
        moderation_model_hc3.params
    ),

    "Robust SE": (
        moderation_model_hc3.bse
    ),

    "t": (
        moderation_model_hc3.tvalues
    ),

    "p-value": (
        moderation_model_hc3.pvalues
    ),

    "95% CI Lower": (
        hc3_confidence_intervals[:, 0]
    ),

    "95% CI Upper": (
        hc3_confidence_intervals[:, 1]
    )
})


rq2_hc3_results["Predictor"] = (
    rq2_hc3_results["Predictor"]
    .map(predictor_labels)
)


print("============================================================")
print("        RQ2 MODERATION — HC3 ROBUSTNESS CHECK               ")
print("============================================================")

display(
    rq2_hc3_results.round(4)
)

In [ ]:
# Bootstrap confidence interval for moderation effect
# The interaction coefficient is bootstrapped to provide an
# additional estimate of uncertainty around the moderation
# effect.
#
# Participants are resampled with replacement. The complete
# moderation model is fitted within each bootstrap sample, and
# the interaction coefficient is recorded.
#
# The 2.5th and 97.5th percentiles form a percentile-based
# 95% bootstrap confidence interval.


def bootstrap_moderation_interaction(
    data,
    n_bootstrap=10000,
    random_seed=42
):
    """
    Bootstrap the Anxiety × AI self-efficacy interaction
    coefficient from the moderation model.
    """

    rng = np.random.default_rng(
        random_seed
    )

    n = len(data)

    interaction_coefficients = []


    for _ in range(n_bootstrap):

        bootstrap_indices = rng.integers(
            0,
            n,
            size=n
        )

        bootstrap_sample = (
            data.iloc[
                bootstrap_indices
            ]
            .copy()
        )


        # Centre variables within the bootstrap sample.
        bootstrap_sample[
            "gad7_centered_boot"
        ] = (
            bootstrap_sample["gad7_total"]
            - bootstrap_sample["gad7_total"].mean()
        )

        bootstrap_sample[
            "aise_centered_boot"
        ] = (
            bootstrap_sample["aise_mean"]
            - bootstrap_sample["aise_mean"].mean()
        )


        bootstrap_sample[
            "interaction_boot"
        ] = (
            bootstrap_sample[
                "gad7_centered_boot"
            ]
            * bootstrap_sample[
                "aise_centered_boot"
            ]
        )


        bootstrap_model = smf.ols(
            formula=(
                "llm_task_mean ~ "
                "gad7_centered_boot + "
                "aise_centered_boot + "
                "interaction_boot"
            ),
            data=bootstrap_sample
        ).fit()


        interaction_coefficients.append(
            bootstrap_model.params[
                "interaction_boot"
            ]
        )


    lower_ci = np.percentile(
        interaction_coefficients,
        2.5
    )

    upper_ci = np.percentile(
        interaction_coefficients,
        97.5
    )


    return lower_ci, upper_ci

# Calculate the bootstrap interaction interval

interaction_boot_ci = (
    bootstrap_moderation_interaction(
        df
    )
)


interaction_coefficient = (
    moderation_model.params[
        "gad7_x_aise"
    ]
)


print("============================================================")
print("          BOOTSTRAP MODERATION EFFECT                       ")
print("============================================================")

print(
    f"Interaction coefficient: "
    f"{interaction_coefficient:.4f}"
)

print(
    f"95% bootstrap CI: "
    f"[{interaction_boot_ci[0]:.4f}, "
    f"{interaction_boot_ci[1]:.4f}]"
)

In [ ]:
# ============================================================
# 15. CONDITIONAL EFFECTS OF ANXIETY AT DIFFERENT LEVELS
#     OF AI SELF-EFFICACY
# ============================================================
# Simple slopes describe the estimated anxiety–LLM-use
# relationship at representative levels of AI self-efficacy:
#
#   Low     = 1 SD below the mean
#   Average = sample mean
#   High    = 1 SD above the mean
#
# These conditional effects help interpret the interaction.
#
# If the interaction itself is not statistically supported,
# the simple slopes should be treated descriptively and should
# not be used to manufacture evidence of moderation.


aise_sd = (
    df["aise_mean"].std()
)


moderator_levels = {
    "Low AI self-efficacy (-1 SD)":
        -aise_sd,

    "Average AI self-efficacy":
        0,

    "High AI self-efficacy (+1 SD)":
        aise_sd
}


# Regression coefficients.
b_anxiety = (
    moderation_model.params[
        "gad7_centered"
    ]
)

b_interaction = (
    moderation_model.params[
        "gad7_x_aise"
    ]
)


simple_slope_results = []


for label, moderator_value in moderator_levels.items():

    conditional_slope = (
        b_anxiety
        + b_interaction
        * moderator_value
    )


    simple_slope_results.append({
        "AI self-efficacy level": label,
        "Centered moderator value": moderator_value,
        "Conditional anxiety slope": conditional_slope
    })


simple_slopes = pd.DataFrame(
    simple_slope_results
)


print("============================================================")
print("              CONDITIONAL ANXIETY SLOPES                    ")
print("============================================================")

display(
    simple_slopes.round(4)
)

# Final validation

In [ ]:
# Final quantitative analysis summary
# This checkpoint confirms completion of the formal
# quantitative analyses used to address RQ1 and RQ2.
#
# RQ1 analyses include:
#
#   - anxiety and overall LLM use
#   - anxiety and mean coding-task LLM use
#   - Pearson/Spearman comparison where appropriate
#   - 95% bootstrap confidence intervals
#   - five task-specific Spearman correlations
#   - Holm correction for multiple comparisons
#
# RQ2 analyses include:
#
#   - mean-centred predictors
#   - Anxiety × AI self-efficacy interaction
#   - multiple linear regression
#   - regression diagnostics
#   - HC3 robust-standard-error sensitivity analysis
#   - bootstrap confidence interval for the interaction
#   - conditional/simple-slope estimates
#
# Final publication-ready tables and figures will be produced
# separately in 05_results_tables_figures.ipynb.


print("============================================================")
print("          QUANTITATIVE ANALYSIS COMPLETE                    ")
print("============================================================")

print(f"\nParticipants analysed: {len(df)}")

print(
    "\nRQ1 primary analyses completed:"
)

print(
    len(rq1_results),
    "primary correlation estimates."
)

print(
    "\nRQ1 task-specific analyses completed:"
)

print(
    len(rq1_task_results),
    "coding-task correlations."
)


interaction_p = (
    moderation_model.pvalues[
        "gad7_x_aise"
    ]
)


print(
    "\nRQ2 interaction coefficient:"
)

print(
    round(
        moderation_model.params[
            "gad7_x_aise"
        ],
        4
    )
)


print(
    "\nRQ2 interaction p-value:"
)

print(
    round(
        interaction_p,
        4
    )
)


print(
    "\n95% bootstrap CI for interaction:"
)

print(
    (
        round(
            interaction_boot_ci[0],
            4
        ),
        round(
            interaction_boot_ci[1],
            4
        )
    )
)


print(
    "\n03_quantitative_analysis.ipynb "
    "completed successfully."
)

# Publication consistency: these are checks, not forced replacements.
expected = [(rho_overall, 0.124), (p_overall, 0.170),
            (r_task_pearson, 0.140), (p_task_pearson, 0.121),
            (rho_task, 0.160), (p_task_spearman, 0.076),
            (moderation_model.params["gad7_x_aise"], -0.010),
            (moderation_model.pvalues["gad7_x_aise"], 0.568)]
for actual, reported in expected:
    if not np.isclose(actual, reported, atol=0.001):
        raise AssertionError(f"Reported-result mismatch: calculated {actual:.5f}; expected about {reported:.3f}")
print("PASS: primary thesis coefficients and p-values match within reporting precision")